# 0.x에서 1.x로 마이그레이션

이 노트북은 Python 3.12 새 커널에서 독립 실행합니다.


## 패키지 설치


In [1]:
# 노트북이 사용하는 패키지 버전을 새 커널 환경에 설치합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 langgraph==1.2.11 python-dotenv==1.2.3 pillow

Note: you may need to restart the kernel to use updated packages.


## 환경 준비


In [2]:
# 공개하지 않은 API 키는 존재 여부만 확인하고 고정한 패키지 버전을 출력합니다.
import os
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env")
        break
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 필요합니다."
print("langchain", version("langchain"))
print("langgraph", version("langgraph"))
print("API 키 존재 여부 확인 완료")

langchain 1.4.0
langgraph 1.2.11
API 키 존재 여부 확인 완료


## 현재 API 서명 확인

마이그레이션은 이름을 일괄 치환하는 작업보다 호출 경계의 변화를 확인하는 작업입니다. `inspect.signature(create_agent)`는 현재 설치된 함수가 받는 매개변수 이름과 기본값을 문자열로 보여 줍니다. 셀의 `for name in [...]` 검사는 이 교재가 사용하는 여섯 인자가 실제 1.4.0 API에 존재하는지 확인하며, 예전 API가 어떻게 동작했는지를 실행하는 검사는 아닙니다.

대응표에서 import 위치, 인자 이름, 반환 상태를 따로 보는 이유가 있습니다. `prompt`를 `system_prompt`로 바꿔도 사용자 상태 형식이나 구조화 결과를 읽는 위치가 예전 코드에 남아 있으면 실행은 일부만 고쳐집니다. 최소 실행은 새 import→그래프 생성→`invoke`→마지막 메시지 읽기까지 한 경로를 통과시켜 호출부가 서로 맞는지 확인합니다.


In [3]:
# 설치된 langchain 1.4.0의 create_agent 서명에서 마이그레이션 대상 인자를 확인합니다.
import inspect
from langchain.agents import create_agent
signature = str(inspect.signature(create_agent))
print(signature)
for name in ["model", "tools", "system_prompt", "response_format", "state_schema", "context_schema"]:
    assert name in signature

(model: 'str | BaseChatModel', tools: 'Sequence[BaseTool | Callable[..., Any] | dict[str, Any]] | None' = None, *, system_prompt: 'str | SystemMessage | None' = None, middleware: 'Sequence[AgentMiddleware[StateT_co, ContextT]]' = (), response_format: 'ResponseFormat[ResponseT] | type[ResponseT] | dict[str, Any] | None' = None, state_schema: 'type[AgentState[ResponseT]] | None' = None, context_schema: 'type[ContextT] | None' = None, checkpointer: 'Checkpointer | None' = None, store: 'BaseStore | None' = None, interrupt_before: 'list[str] | None' = None, interrupt_after: 'list[str] | None' = None, debug: 'bool' = False, name: 'str | None' = None, cache: 'BaseCache[Any] | None' = None, transformers: 'Sequence[TransformerFactory] | None' = None) -> 'CompiledStateGraph[AgentState[ResponseT], ContextT, InputAgentState, OutputAgentState[ResponseT]]'


## 이전과 현재 호출 대조


In [4]:
# 0.x 호출 지점과 v1 대응 지점을 한 행씩 출력해 변경 범위를 분리합니다.
migration = [
 ("create_react_agent", "create_agent", "langgraph.prebuilt에서 langchain.agents로 이동"),
 ("prompt", "system_prompt", "정적 시스템 지시 인자 이름 변경"),
 ("response_format 별도 후처리", "response_format", "에이전트 루프 상태의 structured_response로 반환"),
 ("Pydantic 사용자 상태", "TypedDict 기반 AgentState", "사용자 상태 형식 변경"),
]
for old, new, note in migration: print(f"{old:28} -> {new:28} | {note}")

create_react_agent           -> create_agent                 | langgraph.prebuilt에서 langchain.agents로 이동
prompt                       -> system_prompt                | 정적 시스템 지시 인자 이름 변경
response_format 별도 후처리       -> response_format              | 에이전트 루프 상태의 structured_response로 반환
Pydantic 사용자 상태              -> TypedDict 기반 AgentState      | 사용자 상태 형식 변경


## v1 최소 실행

실제 서비스에서는 먼저 기존 호출의 관찰 테스트를 고정한 뒤 한 경계씩 바꿉니다. 도구 호출 횟수, 메시지 종류, 구조화 결과 필드처럼 사용자에게 보이는 동작을 기록하면 단순히 import가 성공했다는 사실보다 회귀를 빨리 찾을 수 있습니다.

이 절의 표는 대표적인 이동 지점을 설명하며 모든 0.x API의 호환 목록은 아닙니다. 설치된 1.4.0의 서명에 인자가 있다는 사실도 특정 모델 제공자가 모든 조합을 지원한다는 보장은 아닙니다. 구조화 출력과 도구를 함께 쓸 때의 제공자 지원 범위는 02절의 조건을 다시 확인해야 합니다.


In [5]:
# 바꾼 import와 system_prompt를 사용한 최소 에이전트가 실제 응답하는지 검사합니다.
agent = create_agent("openai:gpt-5.4-mini", [], system_prompt="한 문장으로 답하세요.")
result = agent.invoke({"messages": [{"role":"user", "content":"마이그레이션에서 먼저 고정할 것은 무엇인가요?"}]})
print(result["messages"][-1].text)
assert result["messages"][-1].text

마이그레이션에서는 먼저 **현재 시스템의 핵심 요구사항과 데이터/인터페이스의 기준점(스키마·API·동작 규칙)** 을 고정하는 것이 가장 중요합니다.
